# PayGuard Fraud Detection — V2: Synthetic Oversampling (SMOTE)

**Goal:** V1 (`fraud_detection.ipynb`) handled class imbalance with `class_weight='balanced'`.
This notebook tests an alternative: **generating synthetic fraud examples** with SMOTE so the
training set itself is balanced, then training the same model types on that balanced data — to
see whether that beats V1's approach.

**V1 is left completely untouched.** This is a separate, additional experiment; at the end we compare both and adopt whichever performs better.

**Critical rule applied throughout:** synthetic data is generated **only for the training set,
never for the test set**. Evaluating on synthetic test data would mean scoring the model against
fake transactions it was never really facing in production — an easy way to fool yourself into
thinking a model is better than it actually is.

This version uses the standard `imbalanced-learn` library's `SMOTE` implementation (the tool
listed in the assignment's own Resources section), rather than a hand-built version.


## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE   # pip install imbalanced-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix, roc_auc_score
)

pd.set_option('display.max_columns', None)


## 1. Load, Clean, Split — identical to V1

Using the exact same cleaning and time-based split as V1, so the comparison between the two
notebooks is fair (same test set, same train set before any oversampling).

In [3]:
df = pd.read_csv('creditcard.csv')
df = df.drop_duplicates()
df = df.sort_values('Time').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

feature_cols = [c for c in df.columns if c not in ['Class']]
X_train, y_train = train_df[feature_cols].values, train_df['Class'].values
X_test, y_test = test_df[feature_cols].values, test_df['Class'].values

print(f"Train: {X_train.shape}, frauds: {y_train.sum()}")
print(f"Test:  {X_test.shape}, frauds: {y_test.sum()}")


Train: (226980, 30), frauds: 399
Test:  (56746, 30), frauds: 74


## 2. What SMOTE Does

SMOTE (Synthetic Minority Over-sampling Technique) does **not** just copy-paste existing fraud
rows. For each real fraud transaction, it:
1. Finds its `k` nearest fraud neighbors (in feature space).
2. Picks one neighbor at random.
3. Creates a new synthetic point somewhere **on the line between** the original point and that
   neighbor (a random interpolation, not the same point).

This gives new, slightly different fraud-like examples instead of exact duplicates — duplicates
alone would just cause the model to overfit to the same handful of points.

We use `imbalanced-learn`'s built-in `SMOTE` class here, which implements exactly this.

## 3. Apply SMOTE — Training Set Only

By default, `SMOTE()` balances the classes to 50/50 by generating enough synthetic fraud rows to
match the non-fraud count. The test set is **not touched**.

In [4]:
print("Before SMOTE - train class counts:", np.bincount(y_train))

smote = SMOTE(random_state=42)   # random_state fixes the randomness -> reproducible synthetic data
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After SMOTE  - train class counts:", np.bincount(y_train_bal.astype(int)))


Before SMOTE - train class counts: [226581    399]
After SMOTE  - train class counts: [226581 226581]


## 4. Train Models on the Balanced Data

Note: we **don't** use `class_weight='balanced'` here — the data itself is now balanced, so
adding class weighting on top would over-correct. This isolates SMOTE's effect for a clean
comparison against V1.

In [6]:
models = {
    "logistic_regression_smote": LogisticRegression(max_iter=1000, random_state=42),
    "random_forest_smote": RandomForestClassifier(
        n_estimators=200,   
        max_depth=12,
        n_jobs=-1,
        random_state=42
    ),
}

results = {}
for name, model in models.items():
    print(f"=== Training {name} ===")
    model.fit(X_train_bal, y_train_bal)
    proba = model.predict_proba(X_test)[:, 1]  

    pr_auc = average_precision_score(y_test, proba)
    roc_auc = roc_auc_score(y_test, proba)
    print(f"PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f}\n")

    results[name] = {"model": model, "proba": proba, "pr_auc": pr_auc, "roc_auc": roc_auc}


=== Training logistic_regression_smote ===


c:\Users\najaf\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


PR-AUC: 0.7903 | ROC-AUC: 0.9836

=== Training random_forest_smote ===
PR-AUC: 0.8015 | ROC-AUC: 0.9848



## 5. Model Selection — by PR-AUC, same rule as V1

In [7]:
best_name = max(results, key=lambda k: results[k]['pr_auc'])
best = results[best_name]
print(f"Best V2 model: {best_name} (PR-AUC={best['pr_auc']:.4f})")


Best V2 model: random_forest_smote (PR-AUC=0.8015)


## 6. Cost-Based Threshold — same cost assumptions as V1

Using the identical cost figures as V1 ($122 per missed fraud, $5 per wrongly-blocked customer) so
the two notebooks are directly comparable.

In [8]:
COST_FN = 122
COST_FP = 5

precisions, recalls, thresholds = precision_recall_curve(y_test, best['proba'])

best_threshold = 0.5
best_cost = float('inf')
for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):
    preds = (best['proba'] >= t).astype(int)
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    total_cost = fp * COST_FP + fn * COST_FN
    if total_cost < best_cost:
        best_cost = total_cost
        best_threshold = t

print(f"Cost-optimal threshold: {best_threshold:.4f}")
print(f"Estimated total cost at this threshold: ${best_cost:.2f}")


Cost-optimal threshold: 0.4288
Estimated total cost at this threshold: $2060.00


## 7. Final Evaluation

In [9]:
final_preds = (best['proba'] >= best_threshold).astype(int)

print("Classification report:")
print(classification_report(y_test, final_preds, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_test, final_preds))


Classification report:
              precision    recall  f1-score   support

           0     0.9997    0.9992    0.9995     56672
           1     0.5619    0.7973    0.6592        74

    accuracy                         0.9989     56746
   macro avg     0.7808    0.8982    0.8293     56746
weighted avg     0.9992    0.9989    0.9990     56746

Confusion matrix:
[[56626    46]
 [   15    59]]


## 8. V1 vs V2 — Side-by-Side Comparison

**V1 numbers (from `fraud_detection.ipynb`):**

| | **V1 (class_weight='balanced')** |
|---|---|
| Best model | Random Forest |
| PR-AUC | 0.805 |
| ROC-AUC | 0.982 |
| Cost-optimal threshold | 0.107 |
| Recall (fraud caught) | 83.8% (62 / 74) |
| Precision | 39.7% |
| False Positives | ~94 |
| False Negatives | 12 |
| Estimated total cost | ~$1,934 |

**V2 numbers (from `fraud_detection_v2_smote.ipynb`):**

| | **V2 (SMOTE, imbalanced-learn)** |
|---|---|
| Best model | Random Forest |
| PR-AUC | 0.8015 |
| ROC-AUC | 0.9848 |
| Cost-optimal threshold | 0.4288 |
| Recall | 79.7% (59 / 74) |
| Precision | 56.19% |
| False Positives | 46 |
| False Negatives | 15 |
| Estimated total cost | $2,060 |

**Conclusion:** V1 (class_weight) wins — PR-AUC is nearly tied (0.805 vs 0.8015), but V1's total
estimated cost (~$1,934) is lower than V2's ($2,060). V2 cuts false positives roughly in half
(94 → 46), but the extra 3 missed frauds (12 → 15 false negatives) cost more under our $122/$5
assumption than the false-positive savings. **V1 remains the model used for serving** (`main.py`).
